# Mule Account Detection via Gradient Boosting

**Context:** IIT Hyderabad × Bank of India Hackathon (PSB Cybershield) — Problem Statement 2

---

## What is a mule account?

A mule account is a bank account used to receive and forward stolen money. The account holder is often unknowingly recruited — sometimes through fake job ads — and their account becomes a transit node in a money laundering chain.

Detecting mule accounts is genuinely hard:
- They behave *almost like* legitimate accounts, deliberately, to avoid detection.
- Rule-based systems ("flag any account that received a transfer above ₹X") fail because fraudsters adapt to the rules.
- The class imbalance is extreme — mule accounts are rare.

## What this notebook does

This notebook documents the complete pipeline developed for this problem, in order:

1. Load and inspect the raw data
2. Clean the data and remove a temporal leakage feature
3. Perform EDA on hint features provided by the organisers
4. Engineer new features from domain-informed hypotheses
5. Use a Random Forest to rank all features and select the top 50
6. Establish baselines with Logistic Regression and Random Forest
7. Train XGBoost + SMOTE as the primary model
8. Tune the classification threshold using a precision-recall sweep
9. Evaluate on the held-out test set
10. Interpret the model with SHAP

> **Note on feature names:** The dataset uses anonymous feature identifiers (F0–F3924). The organisers provided 18 *hint features* (F115, F321, etc.) as potentially relevant starting points. No semantic labels were given for any feature. Where we have interpretations (e.g., F3898 as "banking relationship depth"), these come from inspecting value distributions by class — they are informed guesses, not confirmed definitions.


## 1. Imports and Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    cross_val_predict,
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Use a non-interactive backend so plots save cleanly
matplotlib.use('Agg')

# ── Constants ──────────────────────────────────────────────────
DATA_FILE      = 'iith boihackathon problem 2 data.csv'
RANDOM_STATE   = 42
TEST_SIZE      = 0.2
CHOSEN_THRESHOLD = 0.45

# Hint features provided by the hackathon organisers
HINTS = [
    'F115', 'F321', 'F527', 'F531', 'F670', 'F1692', 'F2082', 'F2122',
    'F2582', 'F2678', 'F2737', 'F2956', 'F3043', 'F3836', 'F3887',
    'F3889', 'F3891', 'F3894',
]
TARGET = 'F3924'

print('Imports successful.')


## 2. Loading the Data

The dataset is `iith boihackathon problem 2 data.csv`. It must be placed in the same directory as this notebook.

- **9,082 rows** — one per bank account
- **3,925 columns** — features F0–F3923 plus the target F3924
- **Target (`F3924`):** 1 = mule account, 0 = legitimate account


In [ ]:
mydf = pd.read_csv(DATA_FILE)
print(f'Shape: {mydf.shape}')
print(f'\nColumn dtype breakdown:')
print(mydf.dtypes.value_counts())


## 3. The Class Imbalance Problem

Before looking at any features, the first thing to check is the target distribution. The answer here drives every subsequent decision — metric choice, sampling strategy, model selection, and threshold tuning.

**Why ordinary accuracy is inadequate here:**
A model that predicts "legitimate" for every account achieves ~99.1% accuracy. That number is meaningless — it just reflects the proportion of the majority class. The model catches zero mules.

The right metrics for this problem are:
- **Recall (sensitivity):** of all real mules, how many does the model catch?
- **Precision:** of all flagged accounts, how many are actually mules?
- **ROC-AUC:** probability that the model ranks a random mule above a random legitimate account
- **F1:** harmonic mean of precision and recall (used for threshold selection)


In [ ]:
print('Target distribution:')
print(mydf[TARGET].value_counts())
print()
print('As proportions:')
print(mydf[TARGET].value_counts(normalize=True).round(4))

n_mules = mydf[TARGET].sum()
n_legit = (mydf[TARGET] == 0).sum()
ratio = n_legit / n_mules
print(f'\nClass ratio: {ratio:.1f} legitimate accounts per mule')
print(f'Imbalance: {n_mules / len(mydf) * 100:.2f}% mule accounts')


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = mydf[TARGET].value_counts()
ax.bar(['Legitimate (0)', 'Mule (1)'], counts.values,
       color=['steelblue', 'crimson'], edgecolor='white', linewidth=0.8)
for i, v in enumerate(counts.values):
    ax.text(i, v + 30, str(v), ha='center', fontweight='bold')
ax.set_title('Class Distribution: 111 legitimate accounts per mule')
ax.set_ylabel('Number of accounts')
ax.set_yscale('log')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()
print('Saved: class_distribution.png')


## 4. Data Cleaning

### 4.1 Drop columns with >50% missing values

A column with 60% missing values isn't just incomplete — it's actively misleading. Any pattern the model learns from it is a pattern in *which accounts had data recorded*, not the accounts themselves. The 50% threshold is a practical rule of thumb: beyond half-missing, the column has too little data to be trustworthy.


In [ ]:
null_pct = (mydf.isnull().sum() / len(mydf)) * 100
cols_to_drop = null_pct[null_pct > 50].index
print(f'Columns with >50% nulls: {len(cols_to_drop)}')
mydf.drop(columns=cols_to_drop, inplace=True)
print(f'Shape after dropping high-null columns: {mydf.shape}')


### 4.2 Leakage detection and removal — `F2230`

Before any modelling, inspect object-type columns carefully. One of them (`F2230`) contained month-year codes, and inspection revealed a critical problem:


In [ ]:
# Show F2230's distribution by target class
print('F2230 value distribution by class:')
print(mydf.groupby(TARGET)['F2230'].value_counts().to_string())


**The problem:** All 9,001 legitimate accounts have value `Oct25`. The 81 mule accounts are spread across earlier months.

If this feature were kept, the model would learn: *"if F2230 ≠ Oct25, it's probably a mule."* That rule would score near-perfectly on this dataset. But in production, a new incoming account would have whatever the current month is — the feature would mean nothing, and the model's performance would collapse.

This is **data leakage** — information that gives away the label in a way that doesn't generalise to real-world deployment. The temporal feature encodes *when the account was labelled*, not any genuine fraud-risk signal.

The fix is to remove it before any feature engineering or model training.


In [ ]:
# Remove leakage feature and separate the target
X = mydf.drop(columns=['F2230', TARGET])
Y = mydf[TARGET]

print(f'Feature matrix shape after leakage removal: {X.shape}')
print(f'Mules in Y: {Y.sum()} | Legitimate: {(Y==0).sum()}')


### 4.3 Remove near-zero variance features

A feature that is the same value in 99%+ of rows teaches the model nothing — there are no differences to find. These are removed using a variance threshold applied to numeric columns only.


In [ ]:
selector = VarianceThreshold(threshold=0.01)
numeric_cols = X.select_dtypes(exclude='object').columns
X_numeric = X[numeric_cols]

selector.fit(X_numeric)
low_var_cols = numeric_cols[~selector.get_support()]
X.drop(columns=low_var_cols, inplace=True)

print(f'Near-zero variance columns removed: {len(low_var_cols)}')
print(f'Shape after variance filtering: {X.shape}')


### 4.4 Encode categorical features

**`F3888` (account opening date):** Converted to account age in days (`acc_age`). An age-in-days is more informative for a model than a raw date string.

**`F3889` (account age bucket):** Ordinal encoding. The values — `L7D`, `L14D`, `L31D`, `L90D`, `L180D`, `L365D`, `G365D` — have a real order (youngest to oldest). Encoding them as integers 1–7 preserves that order.

**`F3886`, `F3890`–`F3893` (nominal categories):** Label encoding. These have no natural order, but tree-based models split on threshold values so label encoding is fine — the specific integer assigned is irrelevant as long as it's consistent.

**`F2230` was already removed** due to leakage.


In [ ]:
# Convert account date to age in days
X['acc_age'] = (pd.Timestamp.today() - pd.to_datetime(X['F3888'])).dt.days
X.drop(columns=['F3888'], inplace=True)

# Ordinal encoding for account age bucket
age_order = {'L7D': 1, 'L14D': 2, 'L31D': 3, 'L90D': 4,
             'L180D': 5, 'L365D': 6, 'G365D': 7}
X['F3889'] = X['F3889'].map(age_order)

# Label encoding for nominal object columns
le = LabelEncoder()
for col in ['F3886', 'F3890', 'F3891', 'F3892', 'F3893']:
    X[col] = le.fit_transform(X[col].astype(str))

print('Remaining object-type columns after encoding:')
remaining_obj = X.select_dtypes(include='object').columns.tolist()
print(remaining_obj if remaining_obj else 'None')


### 4.5 Impute remaining nulls with column medians

Financial data is right-skewed: a small number of accounts with very large transaction values pull the mean upward. The median is unaffected by outliers and always represents a "typical" account. Median imputation is the standard choice for financial and fraud datasets.


In [ ]:
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns
)

print(f'Nulls remaining after imputation: {X_imputed.isnull().sum().sum()}')
print(f'Feature matrix shape: {X_imputed.shape}')

# Attach target for EDA
plot_df = X_imputed.copy()
plot_df['target'] = Y.values


## 5. Exploratory Data Analysis — Hint Features

The hackathon organisers provided 18 hint features as a starting point. We examine their distributions and class-grouped statistics to understand which ones carry discriminative signal.

**Why histograms alone are insufficient here:** Financial data has extreme right-skew. One account transacting ₹50 crore stretches the x-axis so far that all meaningful variation in the bulk of accounts compresses to a flat line. Side-by-side histograms become visually uninformative even when the signal is real.

We cross-check with a ratio table: `mule mean / legitimate mean`. A ratio far from 1.0 indicates a meaningful class difference even when the plot looks flat.


In [ ]:
# Resolve hint names to post-cleaning equivalents
hints_clean = [f for f in HINTS if f in plot_df.columns]
hints_clean = [('acc_age' if f == 'F3888' else f) for f in hints_clean]
if 'acc_age' not in hints_clean and 'acc_age' in plot_df.columns:
    hints_clean.append('acc_age')
hints_clean = list(dict.fromkeys(hints_clean))  # deduplicate, preserve order

print(f'Hint features available for EDA: {len(hints_clean)}')
print(hints_clean)


In [ ]:
# ── Distribution plots for hint features ──────────────────────
n_hints = len(hints_clean)
n_cols = 3
n_rows = (n_hints + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 3.5))
axes = axes.flatten()

for i, hint in enumerate(hints_clean):
    ax = axes[i]
    plot_df[plot_df['target'] == 0][hint].hist(
        ax=ax, bins=50, alpha=0.6, label='Legitimate', color='steelblue')
    plot_df[plot_df['target'] == 1][hint].hist(
        ax=ax, bins=50, alpha=0.7, label='Mule', color='crimson')
    ax.set_title(hint, fontsize=10)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)

for j in range(n_hints, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Hint Feature Distributions — Legitimate vs Mule', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('hint_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: hint_distributions.png')


In [ ]:
# ── Mean ratio table: mule mean / legitimate mean ─────────────
summary = plot_df.groupby('target')[hints_clean].mean().T
summary.columns = ['Legit (mean)', 'Mule (mean)']
summary['Mule/Legit Ratio'] = summary['Mule (mean)'] / (summary['Legit (mean)'] + 1e-9)
summary = summary.sort_values('Mule/Legit Ratio', ascending=False)
summary = summary.round(4)

print('Hint feature signal summary (sorted by Mule/Legit mean ratio):')
print(summary.to_string())
print()
print('Interpretation: ratio > 1 → mules have higher values; ratio < 1 → mules have lower values')
print('Values furthest from 1.0 indicate stronger class separation.')


In [ ]:
# ── Correlation heatmap among hint features ────────────────────
hints_available = [f for f in hints_clean if f in plot_df.columns]

plt.figure(figsize=(14, 10))
sns.heatmap(
    plot_df[hints_available].corr(),
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0, square=True,
    annot_kws={'size': 7}
)
plt.title('Correlation Between Hint Features')
plt.tight_layout()
plt.savefig('hint_correlation.png', dpi=150)
plt.show()
print('Saved: hint_correlation.png')


**EDA findings:**

| Feature | Signal type | Observation |
|---------|-------------|-------------|
| F2678 | Strong | Mules ~97% lower on average |
| F2122 | Strong | Mules ~9x lower |
| F2956 | Moderate | Mules ~50% lower |
| F2737 | Moderate | Mules ~55% lower |
| F1692 | Moderate | Mules ~57% lower |
| F670 | Moderate | Mules ~2.6x *higher* (the only hint where mules exceed legitimate accounts) |
| F3836 | Weak | Mean misleading due to outliers; medians nearly identical |
| F527, F531, F321, F3894 | Weak | Ratio within 5% of 1.0 |

Notable correlated pairs (possible candidates for ratio features):
- F531 & F321 (r = 0.49)
- F3889 & acc_age (r = 0.41)


## 6. Feature Engineering

### Why feature engineering?

A tree-based model finds patterns by making one yes/no split at a time on one column at a time. Some patterns only exist *across* columns — e.g., a mule might have high F670 *and* low F2678 simultaneously, but either value alone might look unremarkable. Feature engineering pre-computes these cross-column relationships, turning a two-step split into a direct one-step split.

Every engineered feature encodes a hypothesis about the data:
- **Ratio features** capture divergence between related features.
- **Interaction (product) features** amplify signals that point in the same direction for mules.
- **Aggregate features** produce composite scores that are more robust than any single component.
- **Z-score features** flag accounts that are statistically unusual on specific dimensions.

Engineering happens *before* feature selection, so new features compete fairly against the originals.


In [ ]:
# Start from the cleaned, imputed matrix
X_eng = X_imputed.copy()

# ── Ratio features ─────────────────────────────────────────────
# F531 and F321 are correlated (r=0.49). Their ratio captures imbalance between them.
X_eng['F531/F321']    = X_eng['F531'] / (X_eng['F321'] + 1e-6)
X_eng['F3889/acc_age'] = X_eng['F3889'] / (X_eng['acc_age'] + 1e-6)
X_eng['F3894/acc_age'] = X_eng['F3894'] / (X_eng['acc_age'] + 1e-6)

# ── Interaction (product) features ────────────────────────────
# If F2678 and F2122 are both low for mules, their product is very low.
# A legitimate account at (F2678=252, F2122=0.046) has product 11.59.
# A mule at (F2678=2, F2122=0.005) has product 0.01. 1159x difference in one column.
X_eng['F2678*F2122'] = X_eng['F2678'] * X_eng['F2122']
X_eng['F2678*F1692'] = X_eng['F2678'] * X_eng['F1692']
X_eng['F2122*F1692'] = X_eng['F2122'] * X_eng['F1692']

# ── Divergence ratios ─────────────────────────────────────────
# F670 is higher for mules; F2678 is lower. Their ratio should be
# dramatically higher for mules (high numerator ÷ low denominator).
X_eng['F670/F2678'] = X_eng['F670'] / (X_eng['F2678'] + 1e-6)
X_eng['F670/F2122'] = X_eng['F670'] / (X_eng['F2122'] + 1e-6)

# ── Composite aggregate feature ───────────────────────────────
# Five features that were all moderately to strongly lower for mules.
# Their sum provides a single 'account activity' score.
# One noisy component is acceptable; all five simultaneously low is highly diagnostic.
X_eng['sum_of_lows'] = (X_eng['F2678'] + X_eng['F2122'] + X_eng['F2956']
                        + X_eng['F2737'] + X_eng['F1692'])

# ── Z-score features ──────────────────────────────────────────
# Computed across all accounts (majority legitimate), so these measure
# how far each account deviates from the legitimate-account norm.
# Note: these were largely ineffective (see notes), but are included
# here as part of the original pipeline.
for col in ['F2678', 'F2122', 'F670', 'F2956', 'F2737']:
    col_mean = X_eng[col].mean()
    col_std  = X_eng[col].std()
    if col_std > 0:
        X_eng[f'{col}_zscore'] = (X_eng[col] - col_mean) / col_std

new_cols = [c for c in X_eng.columns if c not in X_imputed.columns]
print(f'New features created: {len(new_cols)}')
print(new_cols)
print(f'\nTotal feature count: {X_eng.shape[1]}')
print(f'Nulls in new features: {X_eng[new_cols].isnull().sum().sum()}')


### Why z-scores were largely ineffective here

Z-scores are computed using the mean and standard deviation across all 9,082 accounts. Since 9,001 are legitimate, those statistics are essentially the *legitimate* account statistics. The 81 mules barely shift the distribution. For most features, the mule median z-score ends up similar to the legitimate median z-score — the signal lives in the tails, which don't have enough minority-class mass to move medians.

Z-scores work better when both classes are reasonably balanced or when the minority class is extremely far from the majority. Here, they were included for completeness but feature selection (next section) correctly assigns them low importance.


## 7. Feature Selection

### Why feature selection?

After cleaning and engineering, the feature matrix has ~2,000+ columns. Training XGBoost on all of them would be slow, and — more importantly — many features contain no useful signal. Irrelevant features add noise that the model has to work through, degrading performance on the minority class.

A Random Forest trained on all features ranks them by Gini importance. We keep the top 50 features. This ranking is a rough filter, not ground truth — a feature that doesn't appear in the top 50 might still be useful in combination, but this approach gives a strong starting set.

### Why a Random Forest specifically?

Feature importance from a tree model naturally handles non-linear relationships and interactions. LR-based importance (using absolute coefficients) would be distorted by the scale differences between features. RF Gini importance is a well-understood, computationally efficient ranking method.

**Important caveat:** Feature importance from RF is biased toward features with more unique values (more possible split points). Correlated features also split importance between them. Use this as a rough ranking, not as a definitive importance measurement — that's what SHAP is for.


In [ ]:
# Attach target for the RF
final_dataset = X_eng.copy()
final_dataset['target'] = Y.values

y_sel = final_dataset['target']
X_sel = final_dataset.drop(columns=['target'])

print(f'Feature matrix for selection: {X_sel.shape}')
print(f'Mules: {y_sel.sum()} | Legit: {(y_sel==0).sum()}')


**RF hyperparameter rationale:**

- `n_estimators=500` — more trees = more stable importance rankings
- `max_depth=None` — let trees grow fully to capture complex interactions (OK for ranking, we regularise later for the actual classifier)
- `min_samples_leaf=3, min_samples_split=6` — prevent memorising individual mule accounts
- `class_weight='balanced'` — without this, the RF sees mule samples so rarely that it barely uses them to make splits
- `random_state=42` — reproducibility


In [ ]:
rf_selector = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=3,
    min_samples_split=6,
    max_features='sqrt',
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_selector.fit(X_sel, y_sel)
print('Random Forest training complete.')


In [ ]:
importances = pd.Series(rf_selector.feature_importances_, index=X_sel.columns)
importances = importances.sort_values(ascending=False)

print('Top 30 features by RF Gini importance:')
print(importances.head(30).to_string())


In [ ]:
# Plot top 30 features
plt.figure(figsize=(10, 8))
importances.head(30).plot(kind='barh', color='steelblue')
plt.gca().invert_yaxis()
plt.title('Top 30 Feature Importances (Random Forest, used for selection)')
plt.xlabel('Gini Importance Score')
plt.tight_layout()
plt.savefig('rf_feature_importances.png', dpi=150)
plt.show()
print('Saved: rf_feature_importances.png')


In [ ]:
# Select top 50 features
top_features = importances.head(50).index.tolist()

# Check which engineered features made the top 50
new_cols_check = [
    'F531/F321', 'F3889/acc_age', 'F3894/acc_age',
    'F2678*F2122', 'F2678*F1692', 'F2122*F1692',
    'F670/F2678', 'F670/F2122', 'sum_of_lows',
    'F2678_zscore', 'F2122_zscore', 'F670_zscore',
    'F2956_zscore', 'F2737_zscore',
]
engineered_in_top50 = [c for c in new_cols_check if c in top_features]
print(f'Engineered features that made top 50: {engineered_in_top50}')

# Force-include sum_of_lows (showed real diagnostic signal: median ratio 0.648)
# even if RF didn't rank it in the top 50
X_selected = X_sel[top_features].copy()
if 'sum_of_lows' not in X_selected.columns:
    X_selected['sum_of_lows'] = X_sel['sum_of_lows'].values
    print('sum_of_lows force-added (showed diagnostic signal but missed top 50)')
else:
    print('sum_of_lows already in top 50')

print(f'\nFinal feature set: {X_selected.shape[1]} features')
print(X_selected.columns.tolist())


## 8. Train/Test Split

**Stratification is not optional on this dataset.**

With 81 mules in 9,082 accounts, a naive random split could put 70 mules in training and 11 in test by chance. With 11 mules in the test set, catching one extra mule swings recall by 9 percentage points. The evaluation becomes meaningless.

`stratify=y` forces both splits to maintain the same ~0.89% mule ratio. The actual split produces **~65 mules in training** and **~16 mules in test**.


In [ ]:
X_final = X_selected.copy()
y_final = Y.reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_final
)

print(f'Train: {X_train.shape} | Mules: {y_train.sum()}')
print(f'Test:  {X_test.shape}  | Mules: {y_test.sum()}')
print(f'\nTrain mule rate: {y_train.sum()/len(y_train)*100:.2f}%')
print(f'Test  mule rate: {y_test.sum()/len(y_test)*100:.2f}%')


## 9. Evaluation Setup

A shared evaluation function is used throughout so metrics are computed consistently across all models.

### Reading the confusion matrix

```
                 Predicted: Legit   Predicted: Mule
Actual: Legit  [  True Negative  |  False Positive ]  ← wrongly flagged as mule
Actual: Mule   [  False Negative |  True Positive  ]  ← correctly caught mule
```

In fraud detection, a False Negative (missed mule) is generally worse than a False Positive (investigating a legitimate account). Recall is therefore the primary driver, but precision must remain operationally viable — generating 282 false investigations per 15 genuine catches is unacceptable at scale.

**ROC-AUC** is the most honest single metric for model comparison: it measures ranking quality across all thresholds, not performance at one fixed cutoff.


In [ ]:
def evaluate(model_name, pipeline, X_test, y_test, threshold=0.5, save_cm=True):
    """
    Evaluate a trained pipeline. Returns predicted probabilities.
    threshold: classification threshold (default 0.5; tuned to 0.45 for final model)
    """
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\n{'='*50}")
    print(f'  {model_name} (threshold={threshold})')
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred,
                                target_names=['Legit', 'Mule'], digits=4))
    auc = roc_auc_score(y_test, y_prob)
    print(f'ROC-AUC: {auc:.4f}')

    if save_cm:
        fig, ax = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay.from_predictions(
            y_test, y_pred,
            display_labels=['Legit', 'Mule'],
            cmap='Blues', ax=ax
        )
        fname = f"cm_{model_name.lower().replace(' ', '_').replace('+', 'plus')}"
        fname += f"_thr{str(threshold).replace('.', '')}.png"
        ax.set_title(f'Confusion Matrix — {model_name} (thr={threshold})')
        plt.tight_layout()
        plt.savefig(fname, dpi=150)
        plt.close()
        print(f'Confusion matrix saved: {fname}')

    return y_prob

print('evaluate() helper defined.')


## 10. Baseline — Logistic Regression

Logistic Regression fits a hyperplane through 50-dimensional feature space to divide mule from legitimate accounts. It assumes the boundary is approximately linear — a strong assumption that usually doesn't hold for fraud data. Its role here is to set the floor: any model that can't beat this isn't useful.

**Why a Pipeline?** `StandardScaler` must be fitted only on training data. If the scaler is fitted on the full dataset before splitting, it has seen the test set — a subtle data leak. `Pipeline` enforces the correct order: fit the scaler only on `X_train`, apply the learned scaling to `X_test` at evaluation time.

**Why scale for LR?** LR finds a dividing surface by measuring distances in feature space. A feature measured in millions (some F38xx columns) would dominate a feature measured in single digits (F3889, ordinal 1–7). `StandardScaler` removes units by transforming to mean=0, std=1.

**`class_weight='balanced'`** tells the optimiser that misclassifying a mule is ~111x worse than misclassifying a legitimate account (proportional to the 9001:81 imbalance). Without this, the model would predict "legitimate" for everything and achieve 99.1% accuracy.


In [ ]:
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=RANDOM_STATE,
        C=1.0
    ))
])

lr_pipeline.fit(X_train, y_train)
lr_probs = evaluate('Logistic Regression', lr_pipeline, X_test, y_test, threshold=0.5)


## 11. Baseline — Random Forest

The RF here is the *classifier* baseline, distinct from the RF used in Section 7 for feature selection. The feature-selection RF was never properly evaluated; this one is trained on the final 50-feature set with a proper stratified train/test split.

RF builds 300 decision trees independently, each on a random bootstrap sample and random feature subset. Majority vote is the final prediction. The randomness creates diversity: trees have uncorrelated errors, which cancel when averaged.

**No scaling needed.** Trees split on thresholds — `is F2678 < 0.42?` — and are completely indifferent to feature scale. Adding a scaler would have no effect.

**`class_weight='balanced'`** — same reasoning as LR. Without class weighting, the RF rarely encounters mule accounts in its bootstrap samples.

**`min_samples_leaf=3`** — prevents memorising individual mule accounts. With only 65 mules in training, a fully grown tree would create single-mule leaves. This constraint forces the model to find patterns that generalise across at least 3 accounts.


In [ ]:
rf_pipeline = Pipeline([
    ('clf', RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=3,
        min_samples_split=6,
        max_features='sqrt',
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)
rf_probs = evaluate('Random Forest', rf_pipeline, X_test, y_test, threshold=0.5)


## 12. Cross-Validation on Baseline Models

A single train/test split with 16 mules is too small to trust fully. Whether those 16 happened to be the "easy" or "hard" mules is partly luck. `StratifiedKFold(n_splits=5)` splits the data five ways, trains and evaluates five times, and averages — giving every account one turn in the test set.

High standard deviation on recall is expected here. With ~16 mules per fold, catching one extra mule changes recall by 6.25%. The variance is a consequence of small minority-class counts, not model instability. **Look at ROC-AUC mean for stable model comparison.**


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['recall', 'precision', 'f1', 'roc_auc']

print('=' * 50)
print('  Cross-Validation Results (5-Fold Stratified)')
print('=' * 50)

cv_results = {}
for model_name, pipeline in [
    ('Logistic Regression', lr_pipeline),
    ('Random Forest', rf_pipeline),
]:
    scores = cross_validate(pipeline, X_final, y_final,
                            cv=cv, scoring=scoring, n_jobs=-1)
    cv_results[model_name] = scores
    print(f'\n{model_name}:')
    for metric in scoring:
        vals = scores[f'test_{metric}']
        print(f'  {metric:12s}: {vals.mean():.4f} ± {vals.std():.4f}')


In [ ]:
# ROC curves — both baselines overlaid
fig, ax = plt.subplots(figsize=(7, 6))

for model_name, probs in [('Logistic Regression', lr_probs), ('Random Forest', rf_probs)]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    ax.plot(fpr, tpr, label=f'{model_name} (AUC = {auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', label='Random baseline (AUC = 0.500)')
ax.set_xlabel('False Positive Rate (legitimate accounts wrongly flagged)')
ax.set_ylabel('True Positive Rate (recall — mules actually caught)')
ax.set_title('ROC Curve — Baseline Models')
ax.legend()
plt.tight_layout()
plt.savefig('roc_curve_baselines.png', dpi=150)
plt.show()
print('Saved: roc_curve_baselines.png')


### Interpreting the baseline results

The results look like LR outperforms RF on recall, but this reading is wrong. They make completely different tradeoffs:

| Model | What it actually did |
|-------|---------------------|
| LR | Flagged ~297 accounts. Caught 15/16 mules. 282 false alarms. Precision ≈ 5%. |
| RF | Flagged ~10 accounts. Caught 9/16 mules. ~1 false alarm. Precision ≈ 90%. |

Neither is "better" — they serve different operational tolerance levels. A precision of 5% means 95% of investigations are wasted. A recall of 56% means nearly half of all mules are missed. **The goal of Phase 5 is to find the middle ground: high recall *and* operationally viable precision.**

RF's ROC-AUC of 0.99 (vs LR's 0.97) confirms RF is the better ranker. The default threshold of 0.5 happens to make RF look conservative — lowering it should raise recall while keeping precision far better than LR achieves.


## 13. XGBoost + SMOTE

### Why XGBoost over Random Forest?

Random Forest builds trees independently — each tree is ignorant of what the others got wrong. XGBoost builds trees *sequentially*: each new tree specifically focuses on the accounts the previous trees got wrong. The model iteratively corrects its own mistakes.

This makes XGBoost more powerful on tabular data with subtle, non-linear, multi-feature patterns — exactly what mule detection requires. On standard benchmarks, XGBoost (and its variants) consistently outperform Random Forest on tabular classification. This is a strong prior that held on this dataset.

### Why SMOTE?

With only 65 mules in the training set, the model has limited variation in mule behaviour to learn from. SMOTE (Synthetic Minority Over-sampling Technique) generates synthetic minority class samples: it finds a real mule account, locates its k=5 nearest mule neighbours in feature space, and creates a new fake mule by interpolating between them.

`sampling_strategy=0.3` means mules are 30% of the training set after SMOTE — not 50% (fully balanced). Creating thousands of synthetics from 65 real mules at 50% balance would drift far from the true distribution. 30% gives the model more variation without hallucinating a world where every third account is a mule.

### Why `ImbPipeline` instead of sklearn `Pipeline`?

SMOTE **must only be applied during training, never during prediction**. sklearn's Pipeline doesn't understand this — it would try to run the SMOTE step during `.predict()`. `ImbPipeline` from imbalanced-learn knows that sampling steps only execute during `.fit()`. This is non-negotiable.

### Why not SMOTE + scale_pos_weight together?

Both solve the same problem by different mechanisms: SMOTE rebalances the training distribution; `scale_pos_weight` reweights the loss function. Using both simultaneously over-compensates. The threshold-sweep comparison showed SMOTE alone achieved better precision at equal recall, so `scale_pos_weight` is left at its neutral default of 1.

### XGBoost hyperparameter rationale

| Parameter | Value | Reasoning |
|-----------|-------|----------|
| `n_estimators` | 500 | More trees = more sequential corrections; matched to learning_rate=0.05 |
| `learning_rate` | 0.05 | Small steps; each tree contributes less, requires more trees but reduces overfitting |
| `max_depth` | 4 | With ~65 training mules, deeper trees memorise individual accounts rather than generalising |
| `min_child_weight` | 3 | XGBoost equivalent of `min_samples_leaf`; prevents splits justified by 1–2 mule accounts |
| `subsample` | 0.8 | Each tree trains on 80% of rows — diversity borrowed from RF logic |
| `colsample_bytree` | 0.8 | Each tree sees 80% of features — further reduces overfitting |
| `gamma` | 1.0 | A split is only made if it reduces the loss by ≥1; prevents micro-splits on noise |


In [ ]:
xgb_smote = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),  # safety net for any remaining nulls
    ('smote', SMOTE(
        sampling_strategy=0.3,
        k_neighbors=5,
        random_state=RANDOM_STATE
    )),
    ('xgb', XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=1.0,
        random_state=RANDOM_STATE,
        eval_metric='aucpr',
        verbosity=0,
    ))
])

xgb_smote.fit(X_train, y_train)
print('XGBoost + SMOTE training complete.')


In [ ]:
# Evaluate at default threshold (0.5) — before threshold tuning
y_prob_smote = xgb_smote.predict_proba(X_test)[:, 1]
print('=== XGBoost + SMOTE at default threshold 0.5 (pre-tuning) ===')
_ = evaluate('XGBoost+SMOTE', xgb_smote, X_test, y_test, threshold=0.5, save_cm=False)


## 14. Threshold Tuning

The default threshold of 0.5 is a convention, not a principled choice. It means: *"flag as mule if probability > 50%."* On imbalanced data, this is almost always suboptimal.

Every model outputs a probability score. The threshold determines where we draw the line between "flag" and "don't flag". Moving it:
- **Lower threshold** → catch more mules (↑ recall) but flag more legitimate accounts (↓ precision)
- **Higher threshold** → fewer false alarms (↑ precision) but miss more mules (↓ recall)

The optimal threshold is determined by what the operating context can tolerate — not by convention. We find it empirically by sweeping the threshold from 0.05 to 0.90 and observing precision, recall, and F1 at each step.


In [ ]:
# Threshold sweep
threshold_results = []
for thr in np.arange(0.05, 0.95, 0.05):
    preds = (y_prob_smote >= thr).astype(int)
    threshold_results.append({
        'Threshold': round(thr, 2),
        'Precision': precision_score(y_test, preds, zero_division=0),
        'Recall':    recall_score(y_test, preds),
        'F1':        f1_score(y_test, preds),
    })

threshold_df = pd.DataFrame(threshold_results)
print('Threshold sweep — XGBoost + SMOTE:')
print(threshold_df.round(4).to_string(index=False))


In [ ]:
# Precision-recall-threshold plot
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_smote)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(thresholds, precisions[:-1], label='Precision', color='steelblue', linewidth=2)
ax.plot(thresholds, recalls[:-1],    label='Recall',    color='crimson',   linewidth=2)
ax.axvline(x=0.50, color='gray',   linestyle='--', alpha=0.6, label='Default threshold (0.5)')
ax.axvline(x=CHOSEN_THRESHOLD, color='orange', linestyle='--', linewidth=2,
           label=f'Chosen threshold ({CHOSEN_THRESHOLD}) — P=R crossover')
ax.set_xlabel('Classification Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision and Recall at Every Threshold — XGBoost + SMOTE')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('precision_recall_threshold_xgb_smote.png', dpi=150)
plt.show()
print('Saved: precision_recall_threshold_xgb_smote.png')


### Choosing threshold 0.45

The sweep confirms **0.45** as the optimal operating point:

| Threshold | Precision | Recall | F1 |
|-----------|-----------|--------|----|  
| 0.40 | 0.765 | 0.8125 | 0.788 |
| **0.45** | **0.8125** | **0.8125** | **0.8125** |
| 0.50 | 0.800 | 0.7500 | 0.774 |

At 0.45, precision and recall are exactly equal at 0.8125. This is the crossover point — by definition, also where F1 is maximised (F1 is the harmonic mean of P and R; when P=R, F1=P=R).

**The staircase appearance** in the precision-recall curve is normal: with 16 mules in the test set, each step in recall represents exactly one more mule caught (1/16 = 6.25%). The jagged shape reflects integer counts in the numerator, not model instability.


## 15. Final Evaluation — Test Set (threshold = 0.45)


In [ ]:
# Final predictions at chosen threshold
y_pred_tuned = (y_prob_smote >= CHOSEN_THRESHOLD).astype(int)

print(f'=== XGBoost + SMOTE (threshold={CHOSEN_THRESHOLD}) — FINAL RESULT ===')
print(classification_report(y_test, y_pred_tuned,
                             target_names=['Legitimate', 'Mule'], digits=4))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_smote):.4f}')


In [ ]:
# Confusion matrix at chosen threshold
cm = confusion_matrix(y_test, y_pred_tuned)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_tuned,
    display_labels=['Legitimate', 'Mule'],
    cmap='Blues', ax=ax
)
ax.set_title(f'Confusion Matrix — XGBoost+SMOTE (threshold={CHOSEN_THRESHOLD})')
plt.tight_layout()
plt.savefig('cm_xgboost_smote_final.png', dpi=150)
plt.show()
print('Saved: cm_xgboost_smote_final.png')


In [ ]:
# Operational metrics
print(f"""
Confusion Matrix (threshold={CHOSEN_THRESHOLD}):
                  Predicted Legit   Predicted Mule
Actual Legit      {tn:>14}   {fp:>13}   (False Alarms)
Actual Mule       {fn:>14}   {tp:>13}   (Correctly Caught)

Mules caught:       {tp} / {tp+fn}  ({tp/(tp+fn)*100:.1f}% recall)
False alarms:       {fp} / {tn+fp}  ({fp/(tn+fp)*100:.2f}% of legitimate accounts flagged)
Investigation load: 1 confirmed mule per {(tp+fp)/max(tp,1):.1f} alerts sent to analysts
""")


### Operational interpretation

The *investigation load* metric translates model quality into a practical workflow question: how many accounts does a fraud analyst need to investigate to find one confirmed mule?

At **1.3 alerts per confirmed mule**: investigate 13 accounts, find 10 real mules. That is a viable operational system — far better than the alternative of investigating nothing or relying on rule-based triggers.

A false alarm rate of **0.22%** means that for every 1,000 legitimate accounts, approximately 2 are incorrectly flagged. At the scale of a real bank with millions of accounts, this number matters — but 0.22% is well within acceptable range for a system catching 81% of mules.

**Important caveat:** The test set contains only 16 mule accounts. Each caught or missed mule moves recall by 6.25 percentage points. These test metrics should be interpreted with this uncertainty in mind — see the OOF cross-validation results below for a more robust estimate.


## 16. Out-of-Fold Cross-Validation

The held-out test set contains only 16 mules. Whether those 16 happened to be the easy or hard cases is partly luck.

Out-of-fold (OOF) cross-validation with `cross_val_predict(..., method='predict_proba')` gives every training-set mule account (65 total) exactly one turn in the validation fold from a model that never trained on it. Applying the chosen threshold to these 65 OOF probabilities gives a more reliable estimate of real-world performance.

**OOF is not the same as standard `cross_val_score`.** Standard CV scores at the default 0.5 threshold. OOF with `predict_proba` generates probabilities first, then applies the threshold of 0.45 — evaluating both the model and the threshold choice together.


In [ ]:
cv_oof = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

oof_probs = cross_val_predict(
    xgb_smote, X_train, y_train,
    cv=cv_oof,
    method='predict_proba'
)[:, 1]

oof_preds = (oof_probs >= CHOSEN_THRESHOLD).astype(int)

print(f'=== Out-of-Fold CV Results (threshold={CHOSEN_THRESHOLD}) ===')
print(f'Training set: {len(y_train)} accounts, {y_train.sum()} mules')
print()
print(classification_report(y_train, oof_preds,
                             target_names=['Legitimate', 'Mule'], digits=4))
print(f'OOF ROC-AUC: {roc_auc_score(y_train, oof_probs):.4f}')


In [ ]:
# Summary comparison: test set vs OOF
comparison = pd.DataFrame({
    'Metric':    ['Mule Precision', 'Mule Recall', 'Mule F1', 'ROC-AUC'],
    'Test set (16 mules)': [
        round(precision_score(y_test, y_pred_tuned, zero_division=0), 4),
        round(recall_score(y_test, y_pred_tuned), 4),
        round(f1_score(y_test, y_pred_tuned), 4),
        round(roc_auc_score(y_test, y_prob_smote), 4),
    ],
    'OOF (65 mules)': [
        round(precision_score(y_train, oof_preds, zero_division=0), 4),
        round(recall_score(y_train, oof_preds), 4),
        round(f1_score(y_train, oof_preds), 4),
        round(roc_auc_score(y_train, oof_probs), 4),
    ],
})
print('Test set vs OOF comparison:')
print(comparison.to_string(index=False))
print()
print('The OOF numbers are the more realistic performance expectation.')
print('The test set was slightly favourable (small-sample luck).')


### Why the drop from test set to OOF is expected

The test set showed F1 of 0.8125 for mules. The OOF shows F1 of ~0.76–0.77. The drop is not alarming — with only 16 test mules, the test-set result has high variance. A more honest estimate comes from evaluating all 65 training mules, each exactly once, which is what OOF provides.

The OOF number is the realistic performance expectation on truly unseen data.

### XGBoost vs Random Forest — the OOF comparison resolves the question

On the held-out test set, XGBoost+SMOTE (at 0.45) and Random Forest (at a lower threshold) produced similar confusion matrices — same 13/16 mules caught, same 4 false alarms. With 16 test mules, confusion matrices can only take a small number of distinct values; apparent parity is a sampling artefact.

OOF on all 65 training mules separated them:

| Model | OOF Mule F1 | OOF ROC-AUC |
|-------|-------------|-------------|
| XGBoost + SMOTE | **0.77** | **0.9649** |
| Random Forest   | 0.70    | 0.9538     |

XGBoost+SMOTE outperforms RF on both metrics when evaluated honestly. RF's apparent parity on the test set was noise.


## 17. SHAP Analysis — Model Interpretation

### What SHAP does

SHAP (SHapley Additive exPlanations) breaks each prediction into each feature's contribution. For any individual account:
- **Positive SHAP value** → pushed toward "mule"
- **Negative SHAP value** → pushed toward "legitimate"
- All contributions sum to: final log-odds prediction minus the baseline (average prediction across all accounts)

`TreeExplainer` is used specifically for tree-based models like XGBoost. It computes exact Shapley values, not approximations.

SHAP answers two different questions:
1. **Globally:** which features does the model rely on most across all accounts?
2. **Individually:** for this specific account, which features drove the prediction — and in what direction?


In [ ]:
# Extract the XGBoost model and apply the imputer (SHAP needs imputed data)
xgb_model = xgb_smote.named_steps['xgb']
imputer_step = xgb_smote.named_steps['imputer']

X_test_imputed = imputer_step.transform(X_test)
X_test_imputed = pd.DataFrame(X_test_imputed, columns=X_test.columns)

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_imputed)

print(f'SHAP values shape: {shap_values.shape}')
print(f'Test accounts: {X_test_imputed.shape[0]}, Features: {X_test_imputed.shape[1]}')


In [ ]:
# Global SHAP — mean absolute impact (bar plot)
shap.summary_plot(shap_values, X_test_imputed, plot_type='bar', show=False)
plt.title('SHAP Feature Importance — Mean Absolute Impact')
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: shap_bar.png')


In [ ]:
# Global SHAP — beeswarm (shows direction and magnitude)
shap.summary_plot(shap_values, X_test_imputed, show=False)
plt.title('SHAP Beeswarm — Feature Impact on Mule Prediction')
plt.tight_layout()
plt.savefig('shap_summary_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: shap_summary_beeswarm.png')


### Global findings

**F3898 dominates.** Its mean absolute SHAP value (~2.0) is nearly 3x the next most important feature (F3908 at ~0.75). No other feature comes close.

Top 5 features by global SHAP importance:
1. **F3898** — banking relationship depth (~2.0): legitimate accounts have higher values; mule accounts show systematically shallow histories.
2. **F3908** (~0.75): low values push toward mule.
3. **F270** (~0.52)
4. **F1921** (~0.50)
5. **F2489** (~0.50)

**`sum_of_lows`** appears at position 12 — the engineered composite feature from Section 6 contributed meaningfully to the model's decisions. The feature engineering was not wasted.

**Reading the beeswarm:**
- **F3898:** Red dots (high values) cluster far left (negative SHAP = push toward legitimate). High F3898 strongly protects an account from being flagged. Low F3898 (blue) sits near zero or slightly positive. Translation: legitimate accounts have deep banking relationships; mule accounts don't.
- **F3908:** Blue dots spread right. Low F3908 pushes toward mule; the feature flags mules by *absence*, not high values.
- **sum_of_lows:** Low values push toward mule — consistent with the Phase 2 finding that mule accounts had systematically lower activity on F2678, F2122, F2956, F2737, and F1692.

> Note: Feature semantics are interpretations based on value distributions. The actual meaning of F3898 is not confirmed — the name "banking relationship depth" is an inference from the observation that legitimate accounts cluster at higher values.


In [ ]:
# Individual account analysis — waterfall plots for caught mules
caught_mule_indices = [
    i for i, (true, pred) in enumerate(zip(y_test.values, y_pred_tuned))
    if true == 1 and pred == 1
]

print(f'Caught mules in test set: {len(caught_mule_indices)}')
print(f'Test set row indices (resetting to X_test): {caught_mule_indices}')
print()

# Show waterfall for the most interesting caught mule: Account #1460 (caught by margin of 0.009)
# Find its index in X_test
x_test_reset = X_test.reset_index(drop=True)
y_test_reset  = y_test.reset_index(drop=True)
y_prob_reset  = y_prob_smote
y_pred_reset  = y_pred_tuned

if len(caught_mule_indices) > 0:
    # Print probabilities for all caught mules to show margin of catch
    print('Probabilities for caught mules (threshold=0.45):')
    for idx in caught_mule_indices:
        print(f'  Test index {idx}: p={y_prob_reset[idx]:.4f} (margin from threshold: +{y_prob_reset[idx]-CHOSEN_THRESHOLD:.4f})')


In [ ]:
# Waterfall for the first caught mule
if len(caught_mule_indices) > 0:
    idx = caught_mule_indices[0]
    shap.plots.waterfall(
        shap.Explanation(
            values=shap_values[idx],
            base_values=explainer.expected_value,
            data=X_test_imputed.iloc[idx],
            feature_names=X_test_imputed.columns.tolist()
        ),
        show=False
    )
    plt.title(f'SHAP Waterfall — Caught Mule (test index {idx}, p={y_prob_reset[idx]:.4f})')
    plt.tight_layout()
    plt.savefig(f'shap_waterfall_caught_mule_{idx}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: shap_waterfall_caught_mule_{idx}.png')


In [ ]:
# Waterfall for the missed mule (most informative failure case)
missed_mule_indices = [
    i for i, (true, pred) in enumerate(zip(y_test.values, y_pred_tuned))
    if true == 1 and pred == 0
]

print(f'Missed mules in test set: {len(missed_mule_indices)}')
if missed_mule_indices:
    print('Probabilities for missed mules:')
    for idx in missed_mule_indices:
        print(f'  Test index {idx}: p={y_prob_reset[idx]:.4f} (margin from threshold: {y_prob_reset[idx]-CHOSEN_THRESHOLD:.4f})')

if missed_mule_indices:
    idx_missed = missed_mule_indices[0]
    shap.plots.waterfall(
        shap.Explanation(
            values=shap_values[idx_missed],
            base_values=explainer.expected_value,
            data=X_test_imputed.iloc[idx_missed],
            feature_names=X_test_imputed.columns.tolist()
        ),
        show=False
    )
    plt.title(f'SHAP Waterfall — Missed Mule (test index {idx_missed}, p={y_prob_reset[idx_missed]:.4f})')
    plt.tight_layout()
    plt.savefig('shap_waterfall_missed.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: shap_waterfall_missed.png')


### Individual account analysis — key findings

**The marginal catch (Account #1460 in the original run):**
The model assigned this account a probability of approximately 0.459 — just above the threshold of 0.45. At the default threshold of 0.5, it would have been missed. Threshold tuning from 0.50 to 0.45 is the sole reason this account is in the caught column. The waterfall plot shows F3908=0 pulling strongly toward legitimate (-1.96), overcome by a coalition of smaller features collectively pushing toward mule.

**The missed mule (Account #98 in the original run):**
Two features — F950 (very high transaction volume) and F1921 (high activity) — pulled the prediction strongly toward legitimate, overwhelming signals pointing toward mule. This mule disguised itself through transaction behaviour indistinguishable from a high-activity legitimate account. SHAP explains exactly *why* it was missed: the available features didn't distinguish it.

This failure case points directly to a gap in the available features: **graph-based network signals** (e.g., how many accounts does this account transfer to? what is the average holding time before forwarding?) would likely catch this pattern. No tabular feature in this dataset captures multi-hop transfer behaviour.


## 18. Final Results Summary

In [ ]:
print('=' * 60)
print('FINAL RESULTS — XGBoost + SMOTE (threshold=0.45)')
print('=' * 60)
print()

results_table = pd.DataFrame({
    'Metric': [
        'Model', 'Threshold',
        '--- Test Set (16 mules) ---',
        'Mules Caught', 'Recall', 'Precision', 'F1',
        'False Alarm Rate', 'Investigation Load', 'ROC-AUC',
        '--- OOF CV (65 mules) ---',
        'OOF Recall', 'OOF Precision', 'OOF F1', 'OOF ROC-AUC',
    ],
    'Value': [
        'XGBoost + SMOTE', '0.45',
        '',
        f'{tp} / {tp+fn}',
        f'{tp/(tp+fn)*100:.1f}%',
        f'{tp/(tp+fp)*100:.1f}%',
        f'{f1_score(y_test, y_pred_tuned):.4f}',
        f'{fp/(tn+fp)*100:.2f}%',
        f'1 mule per {(tp+fp)/max(tp,1):.1f} alerts',
        f'{roc_auc_score(y_test, y_prob_smote):.4f}',
        '',
        f'{recall_score(y_train, oof_preds):.4f}',
        f'{precision_score(y_train, oof_preds, zero_division=0):.4f}',
        f'{f1_score(y_train, oof_preds):.4f}',
        f'{roc_auc_score(y_train, oof_probs):.4f}',
    ]
})
print(results_table.to_string(index=False))


In [ ]:
# Baseline comparison table
print('\nBaseline comparison (test set, threshold=0.5):')
baseline_rows = []
for name, pipe, probs in [
    ('Logistic Regression', lr_pipeline, lr_probs),
    ('Random Forest',       rf_pipeline, rf_probs),
]:
    preds = pipe.predict(X_test)
    baseline_rows.append({
        'Model':     name,
        'Precision': round(precision_score(y_test, preds, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, preds), 4),
        'F1':        round(f1_score(y_test, preds), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, probs), 4),
    })

# Add XGB+SMOTE at chosen threshold
baseline_rows.append({
    'Model':     'XGBoost + SMOTE (thr=0.45)',
    'Precision': round(precision_score(y_test, y_pred_tuned, zero_division=0), 4),
    'Recall':    round(recall_score(y_test, y_pred_tuned), 4),
    'F1':        round(f1_score(y_test, y_pred_tuned), 4),
    'ROC-AUC':   round(roc_auc_score(y_test, y_prob_smote), 4),
})

print(pd.DataFrame(baseline_rows).to_string(index=False))


## 19. Limitations

### Small minority class

The test set contains 16 mule accounts. Each caught or missed mule changes recall by 6.25 percentage points. The reported test-set metrics (recall=81.2%, precision=81.25%) carry substantial uncertainty. The OOF estimate over 65 training mules (recall≈74%, F1≈0.76) is more reliable, but still limited by total minority class size.

Any claim about model performance should acknowledge that the confidence intervals on these numbers are wide when the minority class count is this small.

### Anonymous features

Feature names are anonymous (F0–F3924). Interpretations — e.g., F3898 as "banking relationship depth" — are based on inspecting value distributions grouped by class. They are informed inferences, not confirmed definitions. This limits domain-driven model validation.

### No graph/network features

The dataset contains no network information: which accounts transfer to which, how many hops funds travel, average holding time before forwarding, or transaction graph centrality. The missed mule analysis (Account #98) shows that mules disguising themselves through high transaction volume are likely to evade this model — graph-based features would be needed to catch this pattern.

### Temporal leakage was removed but temporal validity is unknown

F2230 was removed because it encoded the labelling month. However, it is unknown whether any remaining features have temporal dependencies that might cause the model to perform differently on future data versus the training window.

### Synthetic minority examples in training

SMOTE generated approximately 2,100 synthetic mule accounts from 65 real ones during training (at sampling_strategy=0.3). Synthetic examples may not represent real mule behaviour patterns not present in the original 65. The model may be less robust to novel mule strategies that weren't in the training data.

### Dataset is a hackathon problem set

This dataset was prepared specifically for this hackathon. Real production bank data would differ in scale, feature distribution, and mule behaviour patterns. Caution is warranted before generalising any conclusions.


## 20. Future Work

These proposals come from specific gaps identified during analysis:

### Graph/network features

The missed mule analysis showed a mule disguised by high transaction volume — indistinguishable from a high-activity legitimate account in tabular form. Transaction network features (transfer graph centrality, average holding time before forwarding, number of recipient accounts, multi-hop transfer detection) would address this pattern. The current feature set has no visibility into the network structure of fund flows.

### Temporal features

F2230 was removed because it constituted leakage in this dataset's construction. But time-based behavioural features that don't encode the label — e.g., change in activity velocity in the past 30 days, rate of new transfer targets — could be genuinely predictive without leaking. These require transaction-level time series, not just account-level aggregates.

### Model calibration

The threshold of 0.45 was selected by sweeping on the same test set used for final evaluation. With only 16 test mules, this introduces a small optimistic bias. A proper calibration approach would use a validation set separate from both training and test sets — difficult with 81 mules total, but worth implementing if more data becomes available.

### Larger minority class

Most of the uncertainty in these results stems from having 81 mule accounts total. Any improvement — additional confirmed mule cases, or a multi-period dataset — would reduce the variance in evaluation metrics and enable more reliable model comparison.
